In [26]:
from typing import TypedDict
from langgraph.graph import StateGraph
from langchain_openai import ChatOpenAI

Breakpoints = controlled pauses in the graph so humans can see, decide, or change things

Main Goals:
1. Approval - Ask user before taking action
2. Debugging - Pause and inspect what went wrong
3. Editing - Change state before continuing



In [27]:
#human approval

from langgraph.graph import StateGraph, END
from typing import TypedDict

# ------------------ STATE ------------------
class State(TypedDict):
    count: int
    approved: bool

# ------------------ NODES ------------------

# Step 1: simple update
def step1(state: State):
    print("Running step1")
    return {"count": state["count"] + 1}

# Step 2: human approval
def approval_node(state: State):
    print("Current State:", state)
    
    decision = input("Approve? (yes/no): ")
    
    if decision == "yes":
        return {"approved": True}
    else:
        return {"approved": False}

# Step 3: final step
def step2(state: State):
    print("Running step2")
    
    if not state["approved"]:
        return {"count": 0}   # reset if rejected
    
    return {"count": state["count"] + 10}


# ------------------ GRAPH ------------------

builder = StateGraph(State)

builder.add_node("step1", step1)
builder.add_node("approval", approval_node)
builder.add_node("step2", step2)

builder.set_entry_point("step1")

builder.add_edge("step1", "approval")
builder.add_edge("approval", "step2")
builder.add_edge("step2", END)

# Breakpoint before approval (for debugging/editing)
graph = builder.compile(interrupt_after=["approval"])

state = {"count": 1, "approved": False}


result = graph.invoke(state)

print("Paused State:", result)

Running step1
Current State: {'count': 2, 'approved': False}


Paused State: {'count': 2, 'approved': True}


In [28]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

class State(TypedDict):
    count: int

def step1(state: State):
    print("Step1")
    return {"count": state["count"] + 1}

def step2(state: State):
    print("Step2")
    return {"count": state["count"] + 10}

builder = StateGraph(State)
builder.add_node("step1", step1)
builder.add_node("step2", step2)

builder.set_entry_point("step1")
builder.add_edge("step1", "step2")
builder.add_edge("step2", END)

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory,
    interrupt_after=["step1"]
)

config = {"configurable": {"thread_id": "1"}}

# First run (pauses)
graph.invoke({"count": 0}, config=config )

# Resume
graph.invoke(None, config=config)

Step1
Step2


{'count': 11}

Editing state = human changes the data produced by the AI before the graph continues execution

In [33]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

#  STATE 
class State(TypedDict):
    amount: int
    approved: bool

# NODE 1: AI GENERATION
def ai_generate(state: State):
    print("AI generating value...")

    # AI produces initial output
    return {
        "amount": 1000,
        "approved": False
    }

# NODE 2: HUMAN FEEDBACK 
def human_review(state: State):
    print("\nCurrent AI Output:", state)

    decision = input("Approve as-is? (yes/no): ")

    if decision.lower() == "yes":
        #  No editing needed
        return {
            "amount": state["amount"],
            "approved": True
        }

    else:
        # Human edits state
        new_amount = int(input("Enter corrected amount: "))

        return {
            "amount": new_amount,
            "approved": True
        }

#  NODE 3
def process(state: State):
    print("\nProcessing with final state:", state)

    if not state["approved"]:
        print("Not approved, stopping.")
        return state

    print(" Payment processed successfully!")
    return state

# ---------------- BUILD GRAPH ----------------
builder = StateGraph(State)

builder.add_node("ai_generate", ai_generate)
builder.add_node("human_review", human_review)
builder.add_node("process", process)

builder.set_entry_point("ai_generate")

builder.add_edge("ai_generate", "human_review")
builder.add_edge("human_review", "process")
builder.add_edge("process", END)

graph = builder.compile()

In [34]:
graph.invoke({"amount": 0, "approved": False})

AI generating value...

Current AI Output: {'amount': 1000, 'approved': False}

Processing with final state: {'amount': 200000, 'approved': True}
 Payment processed successfully!


{'amount': 200000, 'approved': True}